## Hybrid Search LangChain & Pinecone

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY=os.getenv("PINECONE_API_KEY")

In [9]:
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone, ServerlessSpec

index_name="hybrid-search-langchain-pinecone"

## initializing the Pinecone client
pc = Pinecone(api_key=API_KEY)

## create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

In [10]:
index = pc.Index(index_name)
index

In [12]:
from pinecone_text.sparse import BM25Encoder
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [22]:
from langchain_ollama import OllamaEmbeddings
embed=OllamaEmbeddings(model="qwen3-embedding:0.6b")
embed

OllamaEmbeddings(model='qwen3-embedding:0.6b', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [13]:
sentences=[
    "The old clock tower chimed loudly as the fog rolled through the empty streets.",
    "She found a mysterious note tucked inside her favorite book.",
    "A sudden gust of wind scattered papers across the quiet library.",
    "The cat watched the birds with intense curiosity from the window sill.",
    "He decided to take a different route home just to see something new.",
    "The aroma of freshly baked bread filled the small kitchen.",
    "Stars shimmered brightly in the clear night sky above the mountains.",
    "They laughed uncontrollably at a joke no one else understood.",
    "A single drop of rain signaled the beginning of a heavy storm.",
    "The train slowly disappeared into the distance as the sun began to set."
]

## Storing Tf-Idf values
bm25_encoder.fit(sentences)

## store the value to the JSON file
bm25_encoder.dump("bm25_values.json")

## Loading to the BM25Objects
bm25_encoder=BM25Encoder().load("bm25_values.json")

  0%|          | 0/10 [00:00<?, ?it/s]

In [24]:
## Initializing the retriever
retriever=PineconeHybridSearchRetriever(embeddings=embed, sparse_encoder=bm25_encoder, index=index)

In [ ]:
## Adding the sentences to the pinecone index
retriever.add_texts(sentences)

  0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
## Retrieving the relevant sentences for a query
query="What is the aroma that filled the kitchen?"
results=retriever.invoke(query)
results

[Document(metadata={'score': 0.603791475}, page_content='The aroma of freshly baked bread filled the small kitchen.'),
 Document(metadata={'score': 0.190753937}, page_content='She found a mysterious note tucked inside her favorite book.'),
 Document(metadata={'score': 0.19038868}, page_content='The old clock tower chimed loudly as the fog rolled through the empty streets.'),
 Document(metadata={'score': 0.182909012}, page_content='A sudden gust of wind scattered papers across the quiet library.')]

In [27]:
retriever.invoke("What scattered papers across the quiet library?")

[Document(metadata={'score': 0.577550828}, page_content='A sudden gust of wind scattered papers across the quiet library.'),
 Document(metadata={'score': 0.292624474}, page_content='She found a mysterious note tucked inside her favorite book.'),
 Document(metadata={'score': 0.186147213}, page_content='The aroma of freshly baked bread filled the small kitchen.'),
 Document(metadata={'score': 0.176769733}, page_content='The old clock tower chimed loudly as the fog rolled through the empty streets.')]